# Lesson 19: Securing AI Agents with Cryptographic Receipts

In this notebook, you will:
1. Produce **Ed25519 (`PyNaCl`) + RFC 8785 (`jcs`)** signed decision receipts (`agent.tool_call.v1`).
2. Detect byte tampering and attacker re-signing (`04_resigned_receipt`), verify a 3-receipt hash chain, and enforce dual-registry `human.approval.v1` action-digest binding.

In [ ]:
!pip install -q -U pynacl jcs

In [ ]:
import base64, hashlib
from nacl import signing
from jcs import canonicalize

def b64url(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(value: str) -> bytes:
    return base64.urlsafe_b64decode(value + "=" * (-len(value) % 4))

def sha256_canonical(obj) -> str:
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

issuer_key = signing.SigningKey.generate()
pinned_public_key = b64url(bytes(issuer_key.verify_key))
issuer_vk = signing.VerifyKey(b64url_decode(pinned_public_key))

def sign_receipt(payload: dict) -> dict:
    sig = issuer_key.sign(canonicalize(payload)).signature
    return {**payload, "signature": {"alg": "EdDSA", "sig": b64url(sig), "public_key": pinned_public_key}}

def verify_receipt(receipt: dict) -> bool:
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        issuer_vk.verify(canonicalize(payload), b64url_decode(receipt["signature"]["sig"]))
        return True
    except Exception:
        return False

chain = []
previous = None
for sequence, tool_name in enumerate(["lookup_flights", "select_seat", "hold_fare"]):
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": "contoso-travel-bot",
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical({"sequence": sequence}),
        "sequence": sequence,
        "previous_receipt_hash": previous,
    }
    receipt = sign_receipt(payload)
    chain.append(receipt)
    previous = sha256_canonical(receipt)

tampered = {**chain[0], "tool_name": "refund"}
rogue_key = signing.SigningKey.generate()
rogue_payload = {k: v for k, v in tampered.items() if k != "signature"}
rogue_signature = rogue_key.sign(canonicalize(rogue_payload)).signature
rogue = {**rogue_payload, "signature": {"alg": "EdDSA", "sig": b64url(rogue_signature), "public_key": b64url(bytes(rogue_key.verify_key))}}

chain_ok = all(verify_receipt(r) for r in chain)
chain_ok = chain_ok and chain[1]["previous_receipt_hash"] == sha256_canonical(chain[0])
print("Chain:", "VERIFIED" if chain_ok else "REJECTED")
print("Tamper:", "REJECTED" if not verify_receipt(tampered) else "UNEXPECTEDLY ACCEPTED")
print("Attacker re-sign:", "REJECTED" if not verify_receipt(rogue) else "UNEXPECTEDLY ACCEPTED")